In [26]:
# Install modules
!pip install nltk
!pip install transformers[torch]

In [27]:
# Import necessary libraries
import pandas as pd
import nltk
import random
import torch

In [28]:
# Load dataset 
# you can download this dataset from https://huggingface.co/datasets/stepp1/tweet_emotion_intensity/tree/main
data = pd.read_csv('train.csv')

# Preview the data
print(data.head())

      id                                              tweet    class  \
0  40815  Loved @Bethenny independence msg on @WendyWill...     fear   
1  10128  @mark_slifer actually maybe we were supposed t...  sadness   
2  40476  I thought the nausea and headaches had passed ...     fear   
3  20813  Anger, resentment, and hatred are the destroye...    anger   
4  40796  new tires &amp; an alarm system on my car. fwm...     fear   

  sentiment_intensity class_intensity  labels  
0                 low        fear_low       4  
1                high    sadness_high       9  
2              medium     fear_medium       5  
3                high      anger_high       0  
4                 low        fear_low       4  


In [29]:
# Text preprocessing function
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)  # Remove text in square brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabet characters
    return text

# Apply cleaning function
data['cleaned_text'] = data['tweet'].apply(clean_text)

In [10]:
# Tokenize using BERT tokenizer
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokens = tokenizer(data['cleaned_text'].tolist(), max_length=128, padding=True, truncation=True, return_tensors="pt")

In [30]:
from sklearn.model_selection import train_test_split

# Split the dataset into training (70%), validation (15%), and test (15%) sets
train_data, temp_data = train_test_split(data, test_size=0.3, stratify=data['sentiment_intensity'])
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['sentiment_intensity'])

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

Training samples: 2772
Validation samples: 594
Test samples: 594


In [31]:
# If this is your first time using wordnet, download it
nltk.download('wordnet')
from nltk.corpus import wordnet

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/azureuser/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [32]:
def synonym_replacement(word):
    synonyms = wordnet.synsets(word)
    if synonyms:
        return synonyms[0].lemmas()[0].name()
    return word

# Apply synonym replacement on random words in the dataset
def augment_text(text):
    words = text.split()
    augmented_words = [synonym_replacement(word) if random.random() > 0.8 else word for word in words]
    return ' '.join(augmented_words)

# Apply augmentation
data['augmented_text'] = data['cleaned_text'].apply(augment_text)

In [33]:
# Prepare inputs for fine-tuning
from torch.utils.data import DataLoader, TensorDataset
import torch

In [34]:
# Convert inputs and labels into PyTorch tensors
input_ids = tokens['input_ids']
attention_masks = tokens['attention_mask']

# Define a mapping function
def map_sentiment(value):
    if value == "high":
        return 1
    elif value == "medium":
        return 0.5
    elif value == "low":
        return 0
    else:
        return None  # Handle unexpected values, if any

# Apply the function to each item in 'sentiment_intensity'
data['sentiment_intensity'] = data['sentiment_intensity'].apply(map_sentiment)

# Drop any rows where 'sentiment_intensity' is None
data = data.dropna(subset=['sentiment_intensity']).reset_index(drop=True)

# Convert the 'sentiment_intensity' column to a tensor
labels = torch.tensor(data['sentiment_intensity'].tolist())

# Create DataLoader for training
dataset = TensorDataset(input_ids, attention_masks, labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [35]:
data.head()

,id,tweet,class,sentiment_intensity,class_intensity,labels,cleaned_text,augmented_text
0,40815,Loved @Bethenny independence msg on @WendyWill...,fear,0.0,fear_low,4,loved bethenny independence msg on wendywillia...,loved bethenny independence msg on wendywillia...
1,10128,@mark_slifer actually maybe we were supposed t...,sadness,1.0,sadness_high,9,markslifer actually maybe we were supposed to ...,markslifer actually maybe we were supposed to ...
2,40476,I thought the nausea and headaches had passed ...,fear,0.5,fear_medium,5,i thought the nausea and headaches had passed ...,i thought the nausea and headaches had passed ...
3,20813,"Anger, resentment, and hatred are the destroye...",anger,1.0,anger_high,0,anger resentment and hatred are the destroyer ...,anger resentment and hatred are the destroyer ...
4,40796,new tires &amp; an alarm system on my car. fwm...,fear,0.0,fear_low,4,new tires amp an alarm system on my car fwm now,new tires amp an alarm system on my car fwm now
